In [0]:


use databricks_wanderbricks_dataset_dais_2025.wanderbricks;

show tables;

select * from bookings limit 10;

describe  bookings;


-- size and shape

-- 1 total booking record
select count(*) from bookings;

select count(booking_id) from bookings;

select count(distinct(user_id)) from bookings;

-- 
select status, count(booking_id) from bookings group by status;


-- 2 actual booking
select count(booking_id) from bookings;

-- 3 How many properties received bookings
select distinct(status) from bookings;
select count(property_id) from bookings where status = 'confirmed' or status = 'completed';
select status ,count(property_id) from bookings group by status having status = 'confirmed' or status = 'completed';



-- 4 total revenue by property type
create or replace temp view properties_type_view as select p.property_type, b.* from properties p join bookings b on b.property_id = b.property_id;

select * from properties_type_view limit 10;
select property_type, status, round(sum(total_amount),2) as total_revenue from properties_type_view group by property_type, status having status ='completed'; 

-- 5 seasonal trends
select year(check_in)  as year,
month(check_in) as month,
count(*) as total_bookings
from bookings
group by year(check_in), month(check_in)
order by year,month;

select month(check_in) as month, status, count(*) as total_bookings
from bookings
group by month(check_in),status
order by month,status;

SELECT 
    date_format(check_in, 'yyyy-MM') AS month,

    SUM(CASE 
        WHEN status = 'completed' THEN 1 
        ELSE 0 
    END) AS completed,

    SUM(CASE 
        WHEN status = 'cancelled' THEN 1 
        ELSE 0 
    END) AS cancelled,

    SUM(CASE 
        WHEN status = 'pending' THEN 1 
        ELSE 0 
    END) AS pending,

    SUM(CASE 
        WHEN status = 'confirmed' THEN 1 
        ELSE 0 
    END) AS confirmed

FROM bookings

GROUP BY date_format(check_in, 'yyyy-MM')

ORDER BY month;

-- 6 booking status
select distinct status from bookings;

select status, count(*) as booking_count
from bookings
group by status
order by booking_count desc;

-- 7 what time period does booking data cover?
select min(created_at) as first_booking, max(created_at) as latest_booking
from bookings;

-- 8 stay duration
select datediff(check_out, check_in) from bookings;

-- 9 avg guest count by property type/ season
select property_type, round(avg(guests_count),0) as avg_guests from properties_type_view group by property_type; 

-- 10 booking amount
select property_type, round(avg(total_amount),0) as avg_guests from properties_type_view group by property_type; 

-- 11 reapeat booking behaiviar
select * from bookings;
select user_id, count(*) as total_count from bookings group by user_id order by total_count desc;

select total_bookings, count(*) as users
from
(select user_id, count(*) as total_bookings from bookings group by user_id)
group by total_bookings
order by total_bookings;

-- 12 property performance
select p.title, count(*) as total_bookings, 
sum(case when status = 'completed' then 1 else 0 end) as completed_booking,
sum(case when status = 'cancelled' then 1 else 0 end) as cancelled_booking,
round(sum(case when status = 'completed' then total_amount else 0 end),2) as revenue
from bookings b 
join properties p 
on p.property_id = b.property_id
group by p.title
order by revenue desc
Limit 10;



--- Serviced Residence in Singapore generated the highest completed booking revenue
--- Hotel in Singapore and Apartment are also among highest revenue generating properties.
--- For many of the top property cancelled booking are higher.


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.